<a href="https://colab.research.google.com/github/rahimclg/231A005_NLP-Experiments/blob/main/NLPEXP10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from datasets import load_dataset

# Load complete IMDB dataset
dataset = load_dataset("stanfordnlp/imdb")

# Combine train and test
df = pd.concat(
    [dataset["train"].to_pandas(),
     dataset["test"].to_pandas()],
    ignore_index=True
)

# Rename target column
df = df.rename(columns={"label": "labels"})

print(df.shape)
print(df.columns)
print(df["labels"].value_counts())

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

(50000, 2)
Index(['text', 'labels'], dtype='object')
labels
0    25000
1    25000
Name: count, dtype: int64


In [4]:
df.head()

,text,labels
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [5]:


print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())




df["char_count"] = df["text"].astype(str).apply(len)

df["word_count"] = df["text"].astype(str).apply(
    lambda x: len(x.split())
)



Missing values:
text      0
labels    0
dtype: int64

Number of duplicate rows:
418


In [8]:
# Calculate the missing columns first
df["unique_word_count"] = df["text"].astype(str).apply(
    lambda x: len(set(x.split()))
)

df["avg_word_length"] = df["text"].astype(str).apply(
    lambda x: sum(len(word) for word in x.split()) / len(x.split()) if len(x.split()) > 0 else 0
)

# Now describe all four columns
print(
    df[
        [
            "char_count",
            "word_count",
            "unique_word_count",
            "avg_word_length"
        ]
    ].describe()
)

         char_count    word_count  unique_word_count  avg_word_length
count  50000.000000  50000.000000        50000.00000     50000.000000
mean    1309.431020    231.156940          152.14090         4.640676
std      989.728014    171.343997           91.76515         0.340731
min       32.000000      4.000000            4.00000         1.239865
25%      699.000000    126.000000           94.00000         4.417904
50%      970.000000    173.000000          123.00000         4.627006
75%     1590.250000    280.000000          185.00000         4.847458
max    13704.000000   2470.000000         1118.00000        12.290909


In [9]:
df["char_count"] = df["text"].astype(str).apply(len)

df["word_count"] = df["text"].astype(str).apply(
    lambda x: len(x.split())
)

df["unique_word_count"] = df["text"].astype(str).apply(
    lambda x: len(set(x.lower().split()))
)

df["avg_word_length"] = df["text"].astype(str).apply(
    lambda x: sum(len(word) for word in x.split()) /
              max(len(x.split()), 1)
)

print("\nText Statistics:")


Text Statistics:


In [10]:
print(
    df[
        [
            "char_count",
            "word_count",
            "unique_word_count",
            "avg_word_length"
        ]
    ].describe()
)

         char_count    word_count  unique_word_count  avg_word_length
count  50000.000000  50000.000000       50000.000000     50000.000000
mean    1309.431020    231.156940         148.049780         4.640676
std      989.728014    171.343997          88.837364         0.340731
min       32.000000      4.000000           4.000000         1.239865
25%      699.000000    126.000000          91.000000         4.417904
50%      970.000000    173.000000         120.000000         4.627006
75%     1590.250000    280.000000         180.000000         4.847458
max    13704.000000   2470.000000        1092.000000        12.290909


In [12]:
import re

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Keep alphabetic characters only
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Using 'text' instead of the non-existent 'review' column
df["clean_text"] = df["text"].astype(str).apply(clean_text)

In [14]:
from collections import Counter

all_words = " ".join(df["clean_text"]).split()

print("Total number of words:", len(all_words))
print("Vocabulary size:", len(set(all_words)))

word_freq = Counter(all_words)

top_words = word_freq.most_common(20)
print("\nTop 20 most common words:", top_words)

Total number of words: 11709196
Vocabulary size: 99420

Top 20 most common words: [('the', 668009), ('and', 324439), ('a', 323060), ('of', 289415), ('to', 268123), ('is', 211092), ('it', 190862), ('in', 186787), ('i', 175627), ('this', 151001), ('that', 143881), ('s', 129791), ('was', 95606), ('as', 91764), ('movie', 87972), ('for', 87471), ('with', 87368), ('but', 83553), ('film', 79708), ('you', 69131)]
